## Pipeline di Preprocessing - UNSW-NB15

**Obiettivo:** implementare rigorosamente le decisioni metodologiche stabilite durante la fase di EDA per il dataset UNSW-NB15, traducendole in un codice Python robusto, modulare e riutilizzabile.

Questa pipeline applica esclusivamente le trasformazioni approvate nella specifica dei requisiti, senza introdurre alterazioni arbitrarie: nessuna imputazione di valori mancanti, nessuna rimozione di outlier, mantenimento delle feature correlate rilevanti e della variabile `attack_cat`.

In [ ]:
# ==========================================
# 2. IMPORT DELLE LIBRERIE
# ==========================================
import pandas as pd
import numpy as np
from pathlib import Path
from typing import List, Dict, Tuple
import logging

# Configurazione del logging per la tracciabilità delle operazioni
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logging.basicConfig(level=logging.INFO, force=True)

# Import per la gestione del file system in Google Colab
try:
    from google.colab import drive
except ImportError:
    logging.warning("Ambiente Google Colab non rilevato. Assicurarsi di eseguire il codice nell'ambiente corretto se si utilizza Google Drive.")

### Definizione dei Path e Mount di Google Drive

In questa sezione configuriamo i percorsi fisici in base alla struttura del progetto.  
Il mounting di Google Drive garantisce l'accesso al file system persistente del notebook, rendendo disponibili dataset, output e file di supporto tramite il percorso `/content/drive`.

In [ ]:
# ==========================================
# 3. MOUNT DRIVE E DEFINIZIONE PATH
# ==========================================
# Esecuzione del mount di Google Drive
try:
    if not Path("/content/drive/MyDrive").exists():
        logging.info("Montaggio di Google Drive in corso...")
        drive.mount('/content/drive')
    else:
        logging.info("Google Drive e gia montato.")
except Exception as e:
    logging.error(f"Errore durante il mount di Google Drive: {e}")

# Definizione dei percorsi del progetto come costanti pathlib
BASE_PATH = Path("/content/drive/MyDrive/progetto FDSL")
REPO_PATH = BASE_PATH / "unsw-nb15-network-ids"
DATA_PATH = BASE_PATH
TRAIN_PATH = DATA_PATH / "UNSW_NB15_training-set.parquet"
TEST_PATH = DATA_PATH / "UNSW_NB15_testing-set.parquet"
SAVE_PATH = DATA_PATH / "processed_data"

logging.info(f"TRAIN_PATH configurato a: {TRAIN_PATH}")
logging.info(f"TEST_PATH configurato a: {TEST_PATH}")
logging.info(f"SAVE_PATH configurato a: {SAVE_PATH}")


### Caricamento e Verifica Preliminare

Carichiamo il dataset Parquet indicato dal `TRAIN_PATH` e ispezioniamo le dimensioni e le colonne presenti per validare l'integrità iniziale.

In [ ]:
# ==========================================
# 4. CARICAMENTO E VERIFICA DATASET
# ==========================================
def load_dataset(filepath: Path) -> pd.DataFrame:
    """Carica il dataset da file Parquet."""
    if not filepath.exists():
        # Generiamo un DataFrame vuoto per non bloccare brutalmente il notebook,
        # ma segnaliamo l'errore per permettere la correzione del path.
        logging.error(f"File non trovato al percorso: {filepath}")
        return pd.DataFrame()

    logging.info(f"Caricamento dataset da {filepath}...")
    return pd.read_parquet(filepath)

# Caricamento del dataset
df = load_dataset(TRAIN_PATH)

if not df.empty:
    logging.info(f"Shape iniziale del dataset: {df.shape[0]} righe, {df.shape[1]} colonne.")
    logging.info(f"Tipi di dato presenti: \n{df.dtypes.value_counts()}")

### Costanti del Progetto

Qui traduciamo le decisioni specifiche emerse dall'EDA in costanti operative: feature da rimuovere, feature da trasformare logaritmicamente (in base a una skewness > 3), feature da codificare e definizione dei target.

In [ ]:
# ==========================================
# 5. DEFINIZIONE DELLE COSTANTI
# ==========================================

# Feature ridondanti da rimuovere (come da istruzioni esplicite)
FEATURES_TO_DROP = [
    "trans_depth",  # Rimossa al posto di ct_flw_http_mthd
    "is_ftp_login"  # Ridondante rispetto a ct_ftp_cmd
]

# Feature numeriche continue o di conteggio (non binarie) con skewness > 3 che necessitano di trasformazione log1p
# (trans_depth e is_ftp_login sono omesse da questa lista in quanto destinate all'eliminazione)
FEATURES_TO_LOG = [
    "response_body_len", "sbytes", "sloss", "dloss", "dbytes",
    "spkts", "dinpkt", "dpkts", "djit", "sjit",
    "ct_flw_http_mthd", "sinpkt",
    "ct_dst_sport_ltm", "sload", "dur", "ct_ftp_cmd",
    "ct_src_dport_ltm", "rate", "synack", "ackdat", "tcprtt", "dload"
]

# Variabili Categoriche per l'encoding
CATEGORICAL_FEATURES = ["proto", "service", "state"]

# Raggruppamento protocolli rari prima del One-Hot Encoding
PROTO_FEATURE = "proto"
PROTO_RARE_MIN_FREQUENCY = 0.001  # 0.1% del training set dopo deduplicazione
PROTO_RARE_LABEL = "Other"

# Variabile Target e feature intoccabile
TARGET_FEATURE = "label"
UNTOUCHABLE_FEATURE = "attack_cat" # Da mantenere invariata, non usata come target qui


### Funzioni di Preprocessing Modulari

Implementazione rigorosa e "pure-function" (senza side-effects sul DataFrame originale) di tutte le operazioni richieste.  
Ogni funzione possiede un controllo di esistenza delle colonne per garantire l'assenza di crash operativi.

In [ ]:
# ==========================================
# 6. FUNZIONI DI PREPROCESSING
# ==========================================

def verify_dataframe(df: pd.DataFrame, expected_columns: List[str]) -> None:
    """Verifica la presenza delle colonne critiche senza interrompere il flusso."""
    missing = [col for col in expected_columns if col not in df.columns]
    if missing:
        logging.warning(f"Attenzione: Le seguenti colonne attese sono mancanti: {missing}")
    else:
        logging.info("Verifica colonne superata. Tutte le feature attese sono presenti.")

def remove_duplicates(df: pd.DataFrame) -> pd.DataFrame:
    """
    Identifica e rimuove le righe esattamente duplicate nel dataset.
    Decisione: Prevenire distorsioni statistiche e data leakage durante la validazione.
    """
    df_out = df.copy()
    duplicates_count = df_out.duplicated().sum()
    if duplicates_count > 0:
        df_out = df_out.drop_duplicates(ignore_index=True)
        logging.info(f"Rimosse {duplicates_count} righe esattamente duplicate dal dataset.")
    else:
        logging.info("Nessun duplicato esatto rilevato nel dataset.")
    return df_out

def remove_redundant_features(df: pd.DataFrame, columns_to_drop: List[str]) -> pd.DataFrame:
    """
    Rimuove le feature ridondanti.
    Decisione: 'trans_depth' e 'is_ftp_login' vengono rimosse come da istruzioni specifiche.
    """
    df_out = df.copy()
    cols_present = [c for c in columns_to_drop if c in df_out.columns]
    if cols_present:
        df_out = df_out.drop(columns=cols_present)
        logging.info(f"Rimosse {len(cols_present)} feature ridondanti: {cols_present}")
    return df_out

def recode_service_unknown(df: pd.DataFrame) -> pd.DataFrame:
    """
    Ricodifica il valore '-' nella colonna 'service' in 'Unknown'.
    Decisione: Evitare l'imputazione di valori mancanti classici, trattando '-' come categoria semantica.
    """
    df_out = df.copy()
    if 'service' in df_out.columns:
        service_values = df_out['service'].astype(str)
        original_count = int((service_values == '-').sum())
        df_out['service'] = service_values.replace('-', 'Unknown')
        logging.info(f"Ricodificati {original_count} valori '-' in 'Unknown' per la feature 'service'.")
    return df_out

def apply_log_transform(df: pd.DataFrame, columns_to_transform: List[str]) -> pd.DataFrame:
    """
    Applica np.log1p alle feature indicate per mitigare la asimmetria (skewness > 3).
    Decisione: Rispettare in modo rigido l'elenco di feature valutate nell'EDA.
    """
    df_out = df.copy()
    cols_present = [c for c in columns_to_transform if c in df_out.columns]

    for col in cols_present:
        # np.log1p calcola log(1 + x), essenziale per valori che includono lo 0
        df_out[col] = np.log1p(df_out[col].astype(float))

    logging.info(f"Trasformazione log1p applicata a {len(cols_present)} feature numeriche.")
    return df_out

def prepare_categorical_features(df: pd.DataFrame, categorical_features: List[str]) -> pd.DataFrame:
    """
    Esegue l'encoding delle variabili categoriche.
    Decisione: 'state' e 'service' vengono codificate direttamente con One-Hot Encoding;
    'proto' viene prima compattata raggruppando i protocolli rari in una categoria 'Other',
    così da evitare la creazione di troppe colonne dummy poco informative.
    """
    df_out = df.copy()
    cols_present = [c for c in categorical_features if c in df_out.columns]

    for col in cols_present:
        df_out[col] = df_out[col].astype(str)

    if PROTO_FEATURE in cols_present:
        min_count = max(1, int(np.ceil(len(df_out) * PROTO_RARE_MIN_FREQUENCY)))
        proto_counts = df_out[PROTO_FEATURE].value_counts(dropna=False)
        frequent_protocols = proto_counts[proto_counts >= min_count].index
        rare_mask = ~df_out[PROTO_FEATURE].isin(frequent_protocols)
        rare_rows = int(rare_mask.sum())
        rare_categories = int((proto_counts < min_count).sum())

        df_out[PROTO_FEATURE] = df_out[PROTO_FEATURE].where(~rare_mask, PROTO_RARE_LABEL)
        logging.info(
            f"Raggruppati {rare_categories} protocolli rari in '{PROTO_RARE_LABEL}' "
            f"(soglia: almeno {min_count} osservazioni, righe coinvolte: {rare_rows})."
        )

    if cols_present:
        # Applica One-Hot Encoding con prefissi per tracciabilit?
        df_out = pd.get_dummies(df_out, columns=cols_present, drop_first=False, dtype=int)
        logging.info(f"One-Hot Encoding eseguito per le feature: {cols_present}.")
    return df_out

def prepare_numerical_features(df: pd.DataFrame, numeric_features: List[str]) -> pd.DataFrame:
    """
    Verifica e assicura i formati numerici (es. float32/64) per le variabili rimanenti.
    Decisione: Mantenere le colonne numeriche intatte ma strutturalmente sicure senza scalare.
    """
    df_out = df.copy()
    # Si forza il casting a numerico per sicurezza su feature non categoriche e non intoccabili
    target_and_untouchables = [TARGET_FEATURE, UNTOUCHABLE_FEATURE]
    cols_to_check = [c for c in numeric_features if c in df_out.columns and c not in target_and_untouchables]

    for col in cols_to_check:
        df_out[col] = pd.to_numeric(df_out[col], errors='coerce')

    logging.info("Tipi numerici verificati e messi in sicurezza.")
    return df_out

def preprocessing_summary(initial_shape: tuple, final_df: pd.DataFrame) -> None:
    """Genera un riepilogo a console del processo di trasformazione."""
    logging.info("="*40)
    logging.info("📊 RIEPILOGO PREPROCESSING")
    logging.info("="*40)
    logging.info(f"Shape iniziale : {initial_shape}")
    logging.info(f"Shape finale   : {final_df.shape}")
    logging.info(f"Target primario  : '{TARGET_FEATURE}' preservato: {TARGET_FEATURE in final_df.columns}")
    logging.info(f"Colonna intatta  : '{UNTOUCHABLE_FEATURE}' preservata: {UNTOUCHABLE_FEATURE in final_df.columns}")
    logging.info("="*40)


### Esecuzione della Pipeline Sequenziale e Salvataggio

Assembliamo i blocchi precedenti in un'unica pipeline (flusso esecutivo).  
Invocata la procedura, il nuovo dataset viene infine esportato nella directory `SAVE_PATH`.

In [ ]:
# ==========================================
# 7. ESECUZIONE DELLA PIPELINE E SALVATAGGIO
# ==========================================

def run_preprocessing_pipeline(
    df_raw: pd.DataFrame,
    save_path: Path,
    output_filename: str = "UNSW_NB15_training-set_preprocessed.parquet"
) -> pd.DataFrame:
    """
    Invoca in sequenza tutte le funzioni di trasformazione e salva il risultato finale.
    Questa pipeline e usata per il training set, sul quale vengono definite le decisioni di preprocessing.
    """
    if df_raw.empty:
        logging.error("Pipeline interrotta: il DataFrame di input e vuoto.")
        return df_raw

    initial_shape = df_raw.shape
    df_proc = df_raw.copy()

    # 1. Verifica iniziale
    expected = FEATURES_TO_DROP + FEATURES_TO_LOG + CATEGORICAL_FEATURES + [TARGET_FEATURE, UNTOUCHABLE_FEATURE]
    verify_dataframe(df_proc, expected)

    # 2. Rimozione duplicati sul solo training set
    df_proc = remove_duplicates(df_proc)

    # 3. Rimozione feature ridondanti
    df_proc = remove_redundant_features(df_proc, FEATURES_TO_DROP)

    # 4. Ricodifica feature categorica semanticamente non standard
    df_proc = recode_service_unknown(df_proc)

    # 5. Applicazione del logaritmo naturale (+1) per feature con skewness > 3
    df_proc = apply_log_transform(df_proc, FEATURES_TO_LOG)

    # 6. Gestione Feature Categoriche (Encoding)
    df_proc = prepare_categorical_features(df_proc, CATEGORICAL_FEATURES)

    # 7. Preparazione generica feature numeriche
    numeric_candidates = df_proc.select_dtypes(include=[np.number]).columns.tolist()
    df_proc = prepare_numerical_features(df_proc, numeric_candidates)

    # Sintesi a console
    preprocessing_summary(initial_shape, df_proc)

    # Salvataggio su disco (formato Parquet ottimizzato)
    save_path.mkdir(parents=True, exist_ok=True)
    out_file = save_path / output_filename

    try:
        df_proc.to_parquet(out_file, index=False)
        logging.info(f"[OK] Dataset salvato con successo in: {out_file}")
    except Exception as e:
        logging.error(f"[ERRORE] Errore durante il salvataggio del file Parquet: {e}")

    return df_proc

# Avvio reale della pipeline sul training set
if 'df' in locals() and not df.empty:
    processed_df = run_preprocessing_pipeline(df, SAVE_PATH)
    display(processed_df.head())
    display(pd.DataFrame(processed_df.columns, columns=["columns"]))
else:
    logging.warning("Nessun dataset caricato, esecuzione pipeline ignorata.")


## Verifica preventiva del Data Leakage

Prima di applicare la pipeline di preprocessing al test set ufficiale, viene effettuata una verifica per individuare eventuali osservazioni condivise tra il training set e il test set originale.

L'intera pipeline di preprocessing è stata progettata e definita **esclusivamente** sulla base delle analisi condotte sul training set, senza utilizzare alcuna informazione proveniente dal test set. Tuttavia, la presenza di record identici nei due insiemi potrebbe introdurre una forma di **data leakage**, consentendo al modello di essere valutato su esempi già osservati durante la fase di addestramento e producendo una stima eccessivamente ottimistica delle prestazioni.

Per prevenire questo problema, prima dell'applicazione del preprocessing viene eseguito un confronto tra i due dataset originali. Qualora vengano individuate righe comuni, queste vengono rimosse **esclusivamente dal test set**, preservando integralmente il training set utilizzato per l'addestramento dei modelli.

> **Nota:** Questa rappresenta l'unica modifica apportata al test set prima della valutazione finale, oltre all'applicazione della pipeline di preprocessing definita sul training set. Nessuna trasformazione, soglia o parametro del preprocessing viene infatti stimato o adattato utilizzando informazioni provenienti dal test set.

In [ ]:
# ==========================================
# 8. RIMOZIONE DELLE RIGHE COMUNI TRAIN/TEST
# ==========================================

def remove_train_test_overlap(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame
) -> pd.DataFrame:
    """
    Rimuove dal test set le osservazioni identiche presenti anche nel training
    set, prevenendo possibili forme di data leakage.
    """
    logging.info("Verifica di eventuali righe condivise tra training e test...")

    train_unique = train_df.drop_duplicates()
    test_unique = test_df.drop_duplicates()

    common_rows = test_unique.merge(train_unique, how="inner")

    if common_rows.empty:
        logging.info("Nessuna riga condivisa trovata.")
        return test_df

    logging.info(f"Righe condivise individuate: {len(common_rows)}")

    test_without_overlap = (
        test_df
        .merge(common_rows, how="left", indicator=True)
        .query("_merge == 'left_only'")
        .drop(columns="_merge")
    )

    logging.info(f"Righe rimosse dal test: {len(test_df) - len(test_without_overlap)}")

    logging.info(
        f"Dimensione test dopo la rimozione: {len(test_without_overlap)} righe."
    )

    return test_without_overlap

### Applicazione del Preprocessing al Test Set Ufficiale

Il test set ufficiale viene trasformato usando esclusivamente le regole definite sul training set. Non vengono ricalcolate decisioni di preprocessing sul test: le feature rimosse, le feature trasformate con `log1p`, il raggruppamento dei protocolli rari e le colonne one-hot finali sono quelli derivati dal training set preprocessato. Il test set non viene deduplicato, in modo da preservare la partizione ufficiale per la valutazione finale. Tuttavia, eventuali osservazioni condivise con il training set vengono rimosse preventivamente, prima dell'invocazione della pipeline, per prevenire forme di data leakage.


In [ ]:
# ==========================================
# 8. PREPROCESSING DEL TEST SET UFFICIALE
# ==========================================

def get_train_proto_categories(train_columns: List[str]) -> List[str]:
    """Estrae dal training preprocessato le categorie proto mantenute dopo il raggruppamento dei rari."""
    prefix = f"{PROTO_FEATURE}_"
    categories = []
    for col in train_columns:
        if col.startswith(prefix):
            category = col[len(prefix):]
            if category != PROTO_RARE_LABEL:
                categories.append(category)
    return categories


def align_test_categorical_features_to_train(
    df: pd.DataFrame,
    categorical_features: List[str],
    train_columns: List[str]
) -> pd.DataFrame:
    """
    Applica al test set l'encoding categorico compatibile con il training set.
    Le categorie nuove o rare di proto vengono ricondotte a 'Other'; eventuali dummy extra del test vengono scartate.
    """
    df_out = df.copy()
    cols_present = [c for c in categorical_features if c in df_out.columns]

    for col in cols_present:
        df_out[col] = df_out[col].astype(str)

    if PROTO_FEATURE in cols_present:
        train_proto_categories = get_train_proto_categories(train_columns)
        proto_mask = df_out[PROTO_FEATURE].isin(train_proto_categories)
        grouped_rows = int((~proto_mask).sum())
        df_out[PROTO_FEATURE] = df_out[PROTO_FEATURE].where(proto_mask, PROTO_RARE_LABEL)
        logging.info(
            f"Applicato raggruppamento proto al test set: {grouped_rows} righe assegnate a '{PROTO_RARE_LABEL}'."
        )

    if cols_present:
        df_out = pd.get_dummies(df_out, columns=cols_present, drop_first=False, dtype=int)
        logging.info(f"One-Hot Encoding applicato al test set per le feature: {cols_present}.")

    required_columns = [c for c in [TARGET_FEATURE, UNTOUCHABLE_FEATURE] if c in train_columns]
    missing_required = [c for c in required_columns if c not in df_out.columns]
    if missing_required:
        raise ValueError(f"Colonne target mancanti nel test set preprocessato: {missing_required}")

    missing_cols = [c for c in train_columns if c not in df_out.columns]
    for col in missing_cols:
        df_out[col] = 0

    extra_cols = [c for c in df_out.columns if c not in train_columns]
    if extra_cols:
        df_out = df_out.drop(columns=extra_cols)
        logging.info(f"Rimosse {len(extra_cols)} colonne dummy non presenti nel training preprocessato.")

    df_out = df_out[train_columns]
    logging.info(f"Test set allineato alle colonne del training preprocessato: {df_out.shape[1]} colonne.")
    return df_out


def run_test_preprocessing_pipeline(
    df_test_raw: pd.DataFrame,
    train_reference_df: pd.DataFrame,
    save_path: Path,
    output_filename: str = "UNSW_NB15_testing-set_preprocessed.parquet"
) -> pd.DataFrame:
    """
    Applica al test set ufficiale le stesse trasformazioni definite sul training set.
    Il test set non viene usato per decidere soglie, feature o categorie, e non viene deduplicato.
    """
    if df_test_raw.empty:
        logging.error("Pipeline test interrotta: il DataFrame di test e vuoto.")
        return df_test_raw

    if train_reference_df.empty:
        logging.error("Pipeline test interrotta: il training preprocessato di riferimento e vuoto.")
        return df_test_raw

    initial_shape = df_test_raw.shape
    train_columns = train_reference_df.columns.tolist()
    df_test_proc = df_test_raw.copy()

    expected = FEATURES_TO_DROP + FEATURES_TO_LOG + CATEGORICAL_FEATURES + [TARGET_FEATURE, UNTOUCHABLE_FEATURE]
    verify_dataframe(df_test_proc, expected)

    # Non rimuoviamo duplicati dal test ufficiale: deve restare una partizione finale indipendente.
    df_test_proc = remove_redundant_features(df_test_proc, FEATURES_TO_DROP)
    df_test_proc = recode_service_unknown(df_test_proc)
    df_test_proc = apply_log_transform(df_test_proc, FEATURES_TO_LOG)
    df_test_proc = align_test_categorical_features_to_train(df_test_proc, CATEGORICAL_FEATURES, train_columns)

    numeric_candidates = df_test_proc.select_dtypes(include=[np.number]).columns.tolist()
    df_test_proc = prepare_numerical_features(df_test_proc, numeric_candidates)
    df_test_proc = df_test_proc[train_columns]

    preprocessing_summary(initial_shape, df_test_proc)

    save_path.mkdir(parents=True, exist_ok=True)
    out_file = save_path / output_filename

    try:
        df_test_proc.to_parquet(out_file, index=False)
        logging.info(f"[OK] Test set preprocessato salvato con successo in: {out_file}")
    except Exception as e:
        logging.error(f"[ERRORE] Errore durante il salvataggio del test set Parquet: {e}")

    return df_test_proc


# Avvio reale della pipeline sul test set ufficiale
train_reference_path = SAVE_PATH / "UNSW_NB15_training-set_preprocessed.parquet"

if 'processed_df' not in locals() or processed_df.empty:
    processed_df = load_dataset(train_reference_path)

df_test = load_dataset(TEST_PATH)

# Rimozione di eventuali osservazioni condivise
if not df_test.empty and 'df' in locals() and not df.empty:
    df_test = remove_train_test_overlap(df, df_test)

if not df_test.empty and 'processed_df' in locals() and not processed_df.empty:
    processed_test_df = run_test_preprocessing_pipeline(df_test, processed_df, SAVE_PATH)
    display(processed_test_df.head())
    display(pd.DataFrame(processed_test_df.columns, columns=["columns"]))
else:
    logging.warning("Test set o training preprocessato non disponibili, preprocessing del test ignorato.")
